# MultiNest Sampler Tutorial

Based on the MultiNest implementation by Tomas Stolker (2024).

MultiNest is a Bayesian nested sampling algorithm that efficiently explores complex posterior distributions while simultaneously estimating the Bayesian evidence. In contrast to traditional Markov Chain Monte Carlo (MCMC) methods, MultiNest is particularly well suited for multimodal or highly degenerate parameter spaces and enables Bayesian model comparison through evidence estimation.

The `orbitize!` implementation interfaces with the MultiNest library through `PyMultiNest`. Because the native MultiNest library must be compiled separately, users should first complete the MultiNest installation described in the `orbitize!` installation guide before running this tutorial.

In this tutorial we use astrometric measurements of **HD 135344 Ab** to demonstrate the complete workflow:

- loading astrometric observations
- constructing an `orbitize.System`
- configuring the MultiNest sampler
- selecting an appropriate number of live points
- executing the sampler
- saving the posterior samples

## Prerequisites

Before running this tutorial, ensure that:

1. `orbitize!` is installed.
2. MultiNest and `PyMultiNest` are installed (see the installation guide).
3. `astrometry_hd135344ab.csv` is located in the same directory as this notebook.

## Load the Astrometric Data

This tutorial uses relative astrometric measurements of **HD 135344 Ab** provided by Tomas Stolker.

The observations are stored in `astrometry_hd135344ab.csv` and consist of measured right ascension and declination offsets together with their uncertainties and instrument identifiers.

These measurements will be used to construct an `orbitize.System`, which defines the orbital model and likelihood function explored by MultiNest.

Ensure that `astrometry_hd135344ab.csv` is located in the same directory as this notebook before continuing.

The data file is included in the tutorials/ folder of the orbitize! repository.

In [ ]:
from orbitize import read_input
from orbitize import sampler
from orbitize import system

In [ ]:
data_table = read_input.read_file(
    "astrometry_hd135344ab.csv"
)

data_table

The loaded table contains the observation epochs, measured relative astrometry, associated uncertainties, and instrument identifiers.

These measurements define the likelihood function that will be explored by the MultiNest sampler.

## Construct the Orbit System

Next we create an `orbitize.System` object.

This object stores the observational data together with the stellar properties and orbital model that will be sampled by MultiNest.

In [ ]:
orb_system = system.System(
    num_secondary_bodies=1,
    data_table=data_table,
    stellar_or_system_mass=2.2,
    plx=7.41,
    mass_err=0.1,
    plx_err=0.04,
    restrict_angle_ranges=True,
    tau_ref_epoch=58849,
    fit_secondary_mass=False,
    hipparcos_IAD=None,
    gaia=None,
    fitting_basis="Standard",
    use_rebound=False,
)

## Configure the MultiNest Sampler

The principal MultiNest hyperparameter is the number of **live points**.

Increasing the number of live points generally improves posterior accuracy and Bayesian evidence estimation, but also increases computational cost.

200 live points are generally sufficient for testing and tutorial examples.

1000 or more live points are recommended for publication-quality analyses, following the recommendation of Tomas Stolker.

In [ ]:
n_live_points = 200

orb_sampler = sampler.MultiNest(orb_system)

> **Note**
>
> MultiNest requires both the native MultiNest library and the
> `PyMultiNest` Python package.
>
> On Linux, ensure that the directory containing `libmultinest.so`
> is included in `LD_LIBRARY_PATH`.
>
> On macOS, use `DYLD_LIBRARY_PATH`.
>
> If these libraries are unavailable, the sampling step below will fail.
> Please complete the installation instructions before continuing.

In [ ]:
orb_sampler.run_sampler(
    n_live_points=n_live_points,
    output_basename="./multinest/",
    hdf5_file="orbitize.hdf5",
    multinest_kwargs={
        "resume": False
    },
)

## Output

After completion, MultiNest writes the posterior samples to the specified HDF5 file together with the standard MultiNest output directory.

These files can later be analysed using the standard `orbitize!` results utilities without rerunning the sampler.

## Tips for Choosing the Number of Live Points

The number of live points controls the balance between computational cost and sampling quality.

| Live Points | Recommended Usage |
|-------------|-------------------|
| 200 | Quick tutorial / testing |
| 500 | General scientific analyses |
| 1000+ | Publication-quality posterior estimation |

Increasing the number of live points generally improves posterior estimation and Bayesian evidence calculation, although runtime increases approximately linearly.

## Tips for Faster Convergence

Efficient nested sampling depends strongly on the choice of prior distributions and sampler configuration.

The following recommendations have consistently improved convergence when fitting orbital models with `orbitize!`.

## Saving Results

The posterior samples are automatically written to the HDF5 file specified with the `hdf5_file` argument in `run_sampler()`.

This is the recommended approach because the sampler safely handles writing the results, including MPI-enabled executions where writing the file manually after sampling can cause issues.

The saved HDF5 file can later be reloaded with `orbitize!` without rerunning the sampler.

In [ ]:
from orbitize.results import Results

results = Results()

results.load_results("orbitize.hdf5")

print(results.post.shape)

## When should I use MultiNest?

MultiNest is particularly useful when Bayesian evidence is required for model comparison or when the posterior distribution is expected to contain multiple modes.

For simpler orbit-fitting problems where evidence estimation is unnecessary, OFTI or MCMC may provide a more computationally efficient alternative.

## Summary

In this tutorial we:

- loaded astrometric observations of HD 135344 Ab
- constructed an `orbitize.System`
- configured the MultiNest sampler
- discussed the role of the number of live points
- executed the MultiNest sampler
- saved the resulting posterior samples

MultiNest provides an efficient Bayesian nested sampling framework capable of simultaneously estimating posterior distributions and Bayesian evidence, making it particularly valuable for orbit-fitting problems with complex or multimodal parameter spaces.